# Example: 2D data analyzer

Shenyao Jin, shenyaojin@mines.edu

The submodule `fiberis.analyzer.Data2D` is designed to analyze 2D discrete time series data, here is the example how it can read, write and process those kind of data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize

from fiberis.analyzer.Data2D.core2D import Data2D
import datetime

# Data I/O
## Load data
`Data2D` is able to read the formatted npz data file easily using `load_npz` method. In this case, let first load some `HFTS2` data.

In [ ]:
from fiberis.io.reader_hfts2_h5 import HFTS2DAS2D
dataio = HFTS2DAS2D()
dataio.read("../../data/2d/original_data")

In [ ]:
# Load daxis, for original data does not have one.
from fiberis.utils.io_utils import load_hfts2_depthtable
md = load_hfts2_depthtable("../../data/2d/original_data/All_DAS_fibre_depth_table.csvh")

In [ ]:
dataio.daxis = md
DASdata = Data2D()
DASdata = dataio.to_analyzer()

In [ ]:
print(DASdata)

## Write data
`Data2D` provides `savez` method to write a complete `Data2D` object to a npz file. So in the future we can load large amount data without `h5` I/O.

In [ ]:
# Save this dataset to npz for analyze in the future.
# DASdata.savez("../../data/2d/fiberis_format/DASdata_example.npz")
# to load the data, just use
# DASdata.load_npz("../../data/2d/fiberis_format/DASdata_example.npz")

## Merge data

`Data2D` provides a method to merge two DAS datasets, no matter they are continous or not (only need they are not overlapping with each other).

In [ ]:
DASdata_right = DASdata.copy()
DASdata_right.shift(DASdata.taxis[-1])
DASdata.right_merge(DASdata_right) # Here I merged two same DASdatasets.

In [ ]:
# Plot, original data and merged data
# For memory consideration, all operation will change the dataframe itself!
fig, ax = plt.subplots(figsize=(5,5))
im = DASdata_right.plot(ax=ax, cmap='bwr', use_timestamp=True)
im.set_clim(-1000, 1000)
plt.title("DAS dataset example, time shifted")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
im = DASdata.plot(ax=ax, cmap='bwr', use_timestamp=True)
im.set_clim(-1000, 1000)
plt.title("The merged data")
plt.show()

In [ ]:
# Rename the object
DASdata.rename("Merged data")

# Info system
The info system of `Data2D` is very important to keep your research reproducible and reliable. It will record every single operation you did while you call the functions implemented by me. Here is a showcase of the INFO system.

In [ ]:
# Print info
DASdata.history.print_records()

In [ ]:
DASdata.history.add_record("Test record", level='INFO')

In [ ]:
example_filter = lambda record: record.get('level') in ['INFO']
DASdata.history.get_records(example_filter)

# Data processing methods
`Data2D` provides multiple functions to handle the data.

## Select_time
Crop data along the time axis. Modifies data, taxis, and start_time in place. The taxis will be adjusted to start from 0 relative to the new start_time.

In [ ]:
end_time = DASdata.get_end_time()
end_time = end_time - datetime.timedelta(minutes=10)
DASdata_cropped = DASdata.copy()
DASdata_cropped.select_time(DASdata.start_time, end_time)

In [ ]:
# OR you can use sec number to crop the data.
DASdata_cropped_sec = DASdata.copy()
DASdata_cropped_sec.select_time(0, DASdata.taxis[-1] - 600)

In [ ]:
# Plot them for comparison
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
im1 = DASdata_cropped.plot(ax=axs[0], cmap='bwr', use_timestamp=True)
im2 = DASdata_cropped_sec.plot(ax=axs[1], cmap='bwr', use_timestamp=True)
im1.set_clim([-1000, 1000])
im2.set_clim([-1000, 1000])
plt.tight_layout()
plt.show()

**Left**: crop time with `datetime` timestamp.

**Right**: crop time with second number.

## Select depth
Crop data along the depth axis. Modifies data and daxis in place.

> Note: This does not normalize daxis to start from 0.

In [ ]:
DASdata_cropped = DASdata.copy()
DASdata_cropped.select_depth(5000, 10000)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
im = DASdata_cropped.plot(ax=ax, cmap='bwr', use_timestamp=True)
im.set_clim(-1000, 1000)
plt.title("Data waterfall plot after cropping depth")
plt.show()

## Shift start time
Apply a time shift to the data by adjusting start_time.

In [ ]:
DASdata_shift = DASdata.copy()
print("DAS data start time (Original): ", DASdata_shift.start_time)
DASdata_shift.shift(3600)
print("DAS data start time (Shift by 3600 seconds): ", DASdata_shift.start_time)
DASdata_shift.shift(-0.1)
print("DAS data start time (Left shift by 0.1s): ", DASdata_shift.start_time)
DASdata_shift.shift(datetime.timedelta(seconds=0.1))
print("DAS data start time (Right shift by 0.1s): ", DASdata_shift.start_time)

## LP filter
`Data2D` has a low-pass filter using the definition from signal_utils.

In [ ]:
DASdata_filtered = DASdata.copy()
DASdata_filtered.apply_lowpass_filter(0.02) # Note the cutoff freq must be smaller than Nyquist Freq!

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
im = DASdata_filtered.plot(ax=ax, cmap='bwr', use_timestamp=True)
im.set_clim(-1000, 1000)
plt.title("Data waterfall plot after LP filtering")
plt.show()

TBD

In [ ]:
# TBD.